In [29]:
# Neural Identifier Training - Differential Drive Robot
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (Differential Drive Robot)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a differential drive robot.
    state = [x, y, θ, v, ω] 
        x, y: position (m)
        θ: orientation (rad)
        v: linear velocity (m/s)
        ω: angular velocity (rad/s)
    u = [v_cmd, ω_cmd]: velocity commands
    """
    # Robot Parameters
    m = 2.0          # Mass (kg)
    I = 0.5          # Moment of inertia (kg⋅m²)
    b_v = 0.5        # Linear drag coefficient
    b_ω = 0.3        # Angular drag coefficient
    tau_v = 0.2      # Time constant for linear velocity
    tau_ω = 0.15     # Time constant for angular velocity
    
    x, y, θ, v, ω = state
    v_cmd, ω_cmd = u
    
    # Kinematic equations (position and orientation)
    x_dot = v * np.cos(θ)
    y_dot = v * np.sin(θ)
    θ_dot = ω
    
    # Dynamic equations (velocities with first-order dynamics)
    # Modelo simplificado: τ*dv/dt + v = v_cmd
    v_dot = (v_cmd - v) / tau_v - (b_v / m) * v * np.abs(v)  # Con fricción no lineal
    ω_dot = (ω_cmd - ω) / tau_ω - (b_ω / I) * ω * np.abs(ω)  # Con fricción no lineal
    
    return np.array([x_dot, y_dot, θ_dot, v_dot, ω_dot])

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-3):
    """
    RK4 integration step for better accuracy.
    """
    # RK4 para mayor precisión
    k1 = plant_dynamics(x_k, u_k)
    k2 = plant_dynamics(x_k + 0.5*dt*k1, u_k)
    k3 = plant_dynamics(x_k + 0.5*dt*k2, u_k)
    k4 = plant_dynamics(x_k + dt*k3, u_k)
    
    x_kp1 = x_k + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    
    # Add small process noise (Laplacian for heavier tails)
    noise = np.random.normal(0, process_noise_std, size=5)
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=0.5):
    """Sigmoid S(z)."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for Differential Drive Robot - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [x, y, θ, v, ω]
    u_input = [v_cmd, ω_cmd]
    neuron_index: índice de la neurona (0=x, 1=y, 2=θ, 3=v, 4=ω)
    """
    x, y, θ, v, ω = x_est
    v_cmd, ω_cmd = u_input
    
    # Términos básicos sigmoidales
    s_x = sigmoidal(x)
    s_y = sigmoidal(y)
    s_θ = sigmoidal(θ)
    s_v = sigmoidal(v)
    s_ω = sigmoidal(ω)
    
    # Términos trigonométricos (importantes para la cinemática)
    cos_θ = np.cos(θ)
    sin_θ = np.sin(θ)
    
    # Comandos escalados
    s_v_cmd = sigmoidal(v_cmd)
    s_ω_cmd = sigmoidal(ω_cmd)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para x (posición horizontal)
        # dx/dt = v*cos(θ)
        return np.array([
            s_x,                       # Velocidad lineal
            # cos_θ,                     # Componente direccional
            # s_v * cos_θ,              # Término cinemático principal
            s_θ,                       # Orientación
            s_v * s_θ,                # Interacción velocidad-orientación
            s_v**2,                    # Término cuadrático
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 1:  # Neurona para y (posición vertical)
        # dy/dt = v*sin(θ)
        return np.array([
            # s_v,                       # Velocidad lineal
            # sin_θ,                     # Componente direccional
            # s_v * sin_θ,              # Término cinemático principal
            s_θ,                       # Orientación
            s_v * s_θ,                # Interacción velocidad-orientación
            s_y**2,                    # Término cuadrático
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 2:  # Neurona para θ (orientación)
        # dθ/dt = ω
        return np.array([
            s_ω,                       # Velocidad angular (término principal)
            s_ω**2,                    # Término cuadrático
            # s_ω**3,                    # Término cúbico (no linealidad)
            s_v * s_ω,                # Acoplamiento con velocidad lineal
            s_θ,                       # Orientación actual
            # s_ω_cmd * 0.2,            # Comando de velocidad angular
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 3:  # Neurona para v (velocidad lineal)
        # dv/dt = (v_cmd - v)/tau - friction
        return np.array([
            s_v,                       # Estado actual
            s_v**2,                    # Fricción cuadrática
            s_v**3,                    # Fricción cúbica
            # v_cmd * 0.1,              # Comando (lineal, no saturado)
            s_v_cmd,                   # Comando (sigmoidal)
            # s_v * s_v_cmd,            # Interacción estado-comando
            s_ω,                       # Acoplamiento con velocidad angular
            # s_v * np.abs(v),          # Término de fricción absoluta
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 4:  # Neurona para ω (velocidad angular)
        # dω/dt = (ω_cmd - ω)/tau - friction
        return np.array([
            s_ω,                       # Estado actual
            # s_ω**2,                    # Fricción cuadrática
            s_ω**3,                    # Fricción cúbica
            # ω_cmd * 0.1,              # Comando (lineal, no saturado)
            # s_ω_cmd,                   # Comando (sigmoidal)
            s_ω * s_ω_cmd,            # Interacción estado-comando
            s_v,                       # Acoplamiento con velocidad lineal
            # s_ω * np.abs(ω),          # Término de fricción absoluta
            # 1.0                        # Bias
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:  # x
        return 4
    elif neuron_index == 1:  # y
        return 3
    elif neuron_index == 2:  # θ
        return 4
    elif neuron_index == 3:  # v
        return 5
    elif neuron_index == 4:  # ω
        return 4
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-3, R=1e-5):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            # Construir vector de características H_i (ecuación 8)
            z = construct_z_vector(x_k, u_k, i)
            H_i = z.reshape(-1, 1)
            
            # Error de identificación e_i(k) (ecuación 7)
            # e_i(k) = x_i(k) - X̂_i(k)
            x_hat_i = np.dot(self.weights[i], z)
            e_i = x_kp1[i] - x_hat_i
            
            # Ganancia de Kalman K_i(k) (ecuación 6)
            # K_i(k) = P_i(k) H_i(k) [R_i(k) + H_i(k) P_i(k) H_i(k)]^{-1}
            S = self.R + (H_i.T @ self.P[i] @ H_i)[0, 0]
            K_i = (self.P[i] @ H_i).flatten() / S
            
            # Actualización de pesos ω_i(k+1) (ecuación superior)
            # ω_i(k+1) = ω_i(k) + η_i K_i(k) e_i(k)
            self.weights[i] = self.weights[i] + self.eta * K_i * e_i
            
            # Actualización de covarianza P_i(k+1) (ecuación 6, tercera línea)
            # P_i(k+1) = P_i(k) - K_i(k) H_i(k) P_i(k) + Q_i(k)
            self.P[i] = self.P[i] - np.outer(K_i, H_i.flatten()) @ self.P[i] + self.Q_matrices[i]

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.3 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*1e-3 for i in range(n_neurons)]
        self.R = 1e-5
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_particles=1000):  # ✅ Más partículas
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        self.particles = [np.random.randn(n_particles, get_z_size(i))*0.3 
                         for i in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        self.R_std = 0.15
        self.Q_std = 0.057         
        self.regularization_std = 1.0e-5  # ✅ Jitter post-resampling

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n_weights = get_z_size(i)
            
            # 1. Drift con Q_std (no R_std!)
            self.particles[i] += np.random.randn(self.n_particles, n_weights) * self.Q_std
            
            # 2. Weight
            preds = self.particles[i] @ z
            err = x_kp1[i] - preds
            
            # Student-t distribution (heavier tails than Gaussian)
            # log p(y|x) ∝ -log(1 + (err/scale)²)
            nu = 3.0  # degrees of freedom (lower = heavier tails)
            scale = self.R_std * np.sqrt((nu - 2) / nu)  # scale parameter
            log_likelihood = -(nu + 1) / 2 * np.log(1 + (err / scale) ** 2)
            
            self.weights_pf[i] *= np.exp(log_likelihood - np.max(log_likelihood))
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            
            # 3. Resample con regularización
            eff_N = 1.0 / np.sum(self.weights_pf[i]**2)
            if eff_N < self.n_particles/5:  # ✅ Umbral más bajo
                indices = np.random.choice(self.n_particles, self.n_particles, 
                                         p=self.weights_pf[i])
                self.particles[i] = self.particles[i][indices]
                
                # ✅ REGULARIZACIÓN: Agregar jitter
                self.particles[i] += np.random.randn(self.n_particles, n_weights) * self.regularization_std
                
                self.weights_pf[i].fill(1.0/self.n_particles)
                
    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) 
                for i in range(self.n_neurons)]



In [30]:
import time  # Añadir al inicio del archivo

# ============================================================
# 5) Simulation Main Loop - UPDATED FOR PARALLEL CONFIGURATION
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # Main Simulation - PARALLEL CONFIGURATION
    # ============================================================
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 5  # [x, y, θ, v, ω]
    
    # Measurement noise parameters (simulating real sensors)
    position_noise_std = 0.01      # 1 cm position error
    angle_noise_std = 0.005        # 0.5° angle error
    velocity_noise_std = 0.02      # 2 cm/s velocity error
    omega_noise_std = 0.003        # 3°/s angular velocity error
    
    # ============================================================
    # GENERATE INITIAL WEIGHTS (UNIFORM DISTRIBUTION) - SHARED BY ALL FILTERS
    # ============================================================
    np.random.seed(7517)  # For reproducibility
    initial_weights = []
    for i in range(n_states):
        # Uniform distribution U(-1.0, 1.0)
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights.append(w_init.copy())
    
    print("Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}, mean={w.mean():.4f}")
    
    # Initialize Trainers with SAME initial weights
    ekf = EKF_Trainer(n_states, eta=1.0, P0=1.0, Q=1e-3, R=1e-2)
    ukf = UKF_Trainer(n_states, eta=1.0, alpha=1e-2)
    pf = PF_Trainer(n_states, n_particles=600)
    
    # Set initial weights for all filters
    for i in range(n_states):
        ekf.weights[i] = initial_weights[i].copy()
        ukf.weights[i] = initial_weights[i].copy()
        # For particle filter, initialize all particles with the same weights
        pf.particles[i] = np.tile(initial_weights[i], (pf.n_particles, 1))
    
    print("\n✓ All filters initialized with the same RHONN weights")
    
    # Arrays for states
    x_true = np.zeros((n_steps, 5))
    x_est_ekf = np.zeros((n_steps, 5))
    x_est_ukf = np.zeros((n_steps, 5))
    x_est_pf = np.zeros((n_steps, 5))
    
    # Arrays for noisy measurements
    y_measured = np.zeros((n_steps, 5))
    
    # Variables para medir tiempos de entrenamiento
    ekf_training_times = []
    ukf_training_times = []
    pf_training_times = []
    
    # Initial Conditions (robot at origin, facing right)
    x_true[0] = [0.0, 0.0, 0.0, 0.0, 0.0]
    x_est_ekf[0] = x_true[0]
    x_est_ukf[0] = x_true[0]
    x_est_pf[0] = x_true[0]
    
    # Add noise to initial measurement
    initial_noise = np.array([
        np.random.normal(0, position_noise_std),
        np.random.normal(0, position_noise_std),
        np.random.normal(0, angle_noise_std),
        np.random.normal(0, velocity_noise_std),
        np.random.normal(0, omega_noise_std)
    ])
    y_measured[0] = x_true[0] + initial_noise
    
    # Excitation Input
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        v_cmd = 1.0 + 0.5 * np.sin(0.5 * t[k])
        ω_cmd = 0.8 * np.sin(1.0 * t[k])
        
        # Add some step changes for better excitation
        if 300 < k < 320:
            v_cmd = 2.0
            ω_cmd = 1.5
        elif 700 < k < 720:
            v_cmd = 0.5
            ω_cmd = -1.2
        elif 1100 < k < 1120:
            v_cmd = -0.8
            ω_cmd = 0.0
            
        u_hist[k] = [v_cmd, ω_cmd]

    print("\nSimulating Differential Drive Robot with PARALLEL CONFIGURATION...")
    print("\nEstructura de características por neurona:")
    state_names = ['x (pos)', 'y (pos)', 'θ (orient)', 'v (lin vel)', 'ω (ang vel)']
    for i in range(n_states):
        print(f"  Neurona {i} ({state_names[i]}): {get_z_size(i)} características")
    
    print("\nMeasurement Noise Parameters:")
    print(f"  Position (x,y): ±{position_noise_std:.3f} m")
    print(f"  Angle (θ): ±{angle_noise_std:.3f} rad")
    print(f"  Linear velocity (v): ±{velocity_noise_std:.3f} m/s")
    print(f"  Angular velocity (ω): ±{omega_noise_std:.3f} rad/s")
    
    # Main simulation loop with PARALLEL CONFIGURATION
    for k in range(n_steps - 1):
        # 1. Generate true next state
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        
        # 2. Create noisy measurement (simulating real sensors)
        measurement_noise = np.array([
            np.random.laplace(0, position_noise_std/np.sqrt(2)),  # Laplace distribution for position
            np.random.laplace(0, position_noise_std/np.sqrt(2)),  # Laplace distribution for position
            np.random.standard_cauchy() * angle_noise_std * 0.5,  # Cauchy distribution for angle
            np.random.exponential(velocity_noise_std) - velocity_noise_std,  # Exponential (shifted) for velocity
            np.random.uniform(-omega_noise_std*np.sqrt(3), omega_noise_std*np.sqrt(3))  # Uniform for angular velocity
        ])
        y_measured[k+1] = x_true[k+1] + measurement_noise
        
        # 3. Update filters with NOISY MEASUREMENTS (not true states)
        # Each filter uses its OWN previous estimate as input
        
        # --- EKF Update & Predict ---
        start_time = time.perf_counter()
        ekf.update(y_measured[k+1], x_est_ekf[k], u_hist[k])  # Update with measurement
        ekf_time = time.perf_counter() - start_time
        ekf_training_times.append(ekf_time)
        
        # Predict next state using EKF's OWN estimate
        for i in range(5):
            z_ekf = construct_z_vector(x_est_ekf[k], u_hist[k], i)  # PARALLEL: use estimated state
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z_ekf)
        
        # --- UKF Update & Predict ---
        start_time = time.perf_counter()
        ukf.update(y_measured[k+1], x_est_ukf[k], u_hist[k])  # Update with measurement
        ukf_time = time.perf_counter() - start_time
        ukf_training_times.append(ukf_time)
        
        # Predict next state using UKF's OWN estimate
        for i in range(5):
            z_ukf = construct_z_vector(x_est_ukf[k], u_hist[k], i)  # PARALLEL: use estimated state
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z_ukf)
        
        # --- PF Update & Predict ---
        start_time = time.perf_counter()
        pf.update(y_measured[k+1], x_est_pf[k], u_hist[k])  # Update with measurement
        pf_time = time.perf_counter() - start_time
        pf_training_times.append(pf_time)
        
        w_pf = pf.get_estimates()
        # Predict next state using PF's OWN estimate
        for i in range(5):
            z_pf = construct_z_vector(x_est_pf[k], u_hist[k], i)  # PARALLEL: use estimated state
            x_est_pf[k+1, i] = np.dot(w_pf[i], z_pf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            print(f"Step {k}/{n_steps-1}")
            # Show current errors
            err_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
            err_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
            err_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
            print(f"  Current errors - EKF: {err_ekf:.4f}, UKF: {err_ukf:.4f}, PF: {err_pf:.4f}")

    # ============================================================
    # 6) Calculate Statistics - UPDATED FOR FAIR COMPARISON
    # ============================================================
    
    print("\n" + "="*70)
    print("📊 RESULTS - PARALLEL CONFIGURATION WITH MEASUREMENT NOISE")
    print("="*70)
    
    # Calculate MSE against TRUE states (not measurements)
    mse_x_ekf = np.mean((x_true[:, 0] - x_est_ekf[:, 0])**2)
    mse_y_ekf = np.mean((x_true[:, 1] - x_est_ekf[:, 1])**2)
    mse_θ_ekf = np.mean((x_true[:, 2] - x_est_ekf[:, 2])**2)
    mse_v_ekf = np.mean((x_true[:, 3] - x_est_ekf[:, 3])**2)
    mse_ω_ekf = np.mean((x_true[:, 4] - x_est_ekf[:, 4])**2)
    
    mse_x_ukf = np.mean((x_true[:, 0] - x_est_ukf[:, 0])**2)
    mse_y_ukf = np.mean((x_true[:, 1] - x_est_ukf[:, 1])**2)
    mse_θ_ukf = np.mean((x_true[:, 2] - x_est_ukf[:, 2])**2)
    mse_v_ukf = np.mean((x_true[:, 3] - x_est_ukf[:, 3])**2)
    mse_ω_ukf = np.mean((x_true[:, 4] - x_est_ukf[:, 4])**2)
    
    mse_x_pf = np.mean((x_true[:, 0] - x_est_pf[:, 0])**2)
    mse_y_pf = np.mean((x_true[:, 1] - x_est_pf[:, 1])**2)
    mse_θ_pf = np.mean((x_true[:, 2] - x_est_pf[:, 2])**2)
    mse_v_pf = np.mean((x_true[:, 3] - x_est_pf[:, 3])**2)
    mse_ω_pf = np.mean((x_true[:, 4] - x_est_pf[:, 4])**2)
    
    # Total MSE for comparison
    mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_θ_ekf + mse_v_ekf + mse_ω_ekf
    mse_total_ukf = mse_x_ukf + mse_y_ukf + mse_θ_ukf + mse_v_ukf + mse_ω_ukf
    mse_total_pf = mse_x_pf + mse_y_pf + mse_θ_pf + mse_v_pf + mse_ω_pf
    
    # Report
    print("\n--- Comparación de Desempeño (MSE contra estado REAL) ---")
    print(f"EKF MSE total: {mse_total_ekf:.6f}")
    print(f"  x: {mse_x_ekf:.6f} | y: {mse_y_ekf:.6f} | θ: {mse_θ_ekf:.6f} | v: {mse_v_ekf:.6f} | ω: {mse_ω_ekf:.6f}")
    print(f"UKF MSE total: {mse_total_ukf:.6f}")
    print(f"  x: {mse_x_ukf:.6f} | y: {mse_y_ukf:.6f} | θ: {mse_θ_ukf:.6f} | v: {mse_v_ukf:.6f} | ω: {mse_ω_ukf:.6f}")
    print(f"PF  MSE total: {mse_total_pf:.6f}")
    print(f"  x: {mse_x_pf:.6f} | y: {mse_y_pf:.6f} | θ: {mse_θ_pf:.6f} | v: {mse_v_pf:.6f} | ω: {mse_ω_pf:.6f}")
    
    # Análisis de tiempos de entrenamiento
    print("\n" + "="*70)
    print("⏱️  TIEMPOS DE ENTRENAMIENTO POR FILTRO")
    print("="*70)
    
    # Calcular estadísticas de tiempos
    ekf_total_time = np.sum(ekf_training_times)
    ukf_total_time = np.sum(ukf_training_times)
    pf_total_time = np.sum(pf_training_times)
    
    ekf_mean_time = np.mean(ekf_training_times)
    ukf_mean_time = np.mean(ukf_training_times)
    pf_mean_time = np.mean(pf_training_times)
    
    ekf_std_time = np.std(ekf_training_times)
    ukf_std_time = np.std(ukf_training_times)
    pf_std_time = np.std(pf_training_times)
    
    # Encontrar el más rápido y más lento
    fastest_mean = min(ekf_mean_time, ukf_mean_time, pf_mean_time)
    slowest_mean = max(ekf_mean_time, ukf_mean_time, pf_mean_time)
    
    print(f"\n📈 Estadísticas de Tiempos por Iteración (en segundos):")
    print(f"\nEKF-RHONN:")
    print(f"  Total: {ekf_total_time:.6f} s | Media: {ekf_mean_time*1000:.3f} ms")
    print(f"  Std: {ekf_std_time*1000:.3f} ms | Min: {np.min(ekf_training_times)*1000:.3f} ms")
    print(f"  Max: {np.max(ekf_training_times)*1000:.3f} ms")
    
    print(f"\nUKF-RHONN:")
    print(f"  Total: {ukf_total_time:.6f} s | Media: {ukf_mean_time*1000:.3f} ms")
    print(f"  Std: {ukf_std_time*1000:.3f} ms | Min: {np.min(ukf_training_times)*1000:.3f} ms")
    print(f"  Max: {np.max(ukf_training_times)*1000:.3f} ms")
    
    print(f"\nPF-RHONN:")
    print(f"  Total: {pf_total_time:.6f} s | Media: {pf_mean_time*1000:.3f} ms")
    print(f"  Std: {pf_std_time*1000:.3f} ms | Min: {np.min(pf_training_times)*1000:.3f} ms")
    print(f"  Max: {np.max(pf_training_times)*1000:.3f} ms")
    
    # Comparación relativa
    print(f"\n⚡ Comparación Relativa de Velocidad:")
    print(f"  EKF es {ukf_mean_time/ekf_mean_time:.2f}x más rápido que UKF")
    print(f"  EKF es {pf_mean_time/ekf_mean_time:.2f}x más rápido que PF")
    print(f"  UKF es {pf_mean_time/ukf_mean_time:.2f}x más rápido que PF")
    
    # Calcular eficiencia (MSE por unidad de tiempo)
    print(f"\n🎯 Eficiencia (MSE / Tiempo de Entrenamiento):")
    print(f"  EKF: {mse_total_ekf/ekf_total_time:.6f} MSE/s")
    print(f"  UKF: {mse_total_ukf/ukf_total_time:.6f} MSE/s")
    print(f"  PF:  {mse_total_pf/pf_total_time:.6f} MSE/s")
    
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    mse_dict = {'EKF-RHONN': mse_total_ekf, 'UKF-RHONN': mse_total_ukf, 'PF-RHONN': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
    print("="*70)

Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(4,), min=-0.7101, max=0.3165, mean=-0.1913
  Neuron 1: shape=(3,), min=-0.6689, max=0.9859, mean=-0.0875
  Neuron 2: shape=(4,), min=-0.7763, max=0.7724, mean=-0.2666
  Neuron 3: shape=(5,), min=-0.8238, max=0.7480, mean=0.0252
  Neuron 4: shape=(4,), min=-0.7120, max=0.3356, mean=-0.0417

✓ All filters initialized with the same RHONN weights

Simulating Differential Drive Robot with PARALLEL CONFIGURATION...

Estructura de características por neurona:
  Neurona 0 (x (pos)): 4 características
  Neurona 1 (y (pos)): 3 características
  Neurona 2 (θ (orient)): 4 características
  Neurona 3 (v (lin vel)): 5 características
  Neurona 4 (ω (ang vel)): 4 características

Measurement Noise Parameters:
  Position (x,y): ±0.010 m
  Angle (θ): ±0.005 rad
  Linear velocity (v): ±0.020 m/s
  Angular velocity (ω): ±0.003 rad/s
Step 300/999
  Current errors - EKF: 0.0393, UKF: 0.0253, PF: 0.0209
Step 600/999
  Current errors - EKF: 0.0

In [31]:
    # ============================================================
    # 7) Visualization - Updated for realistic results
    # ============================================================
    
    print("\nGenerando visualizaciones...")
    
    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'line_width_meas': 1.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }
    
    # --- Gráfica de tiempos de entrenamiento ---
    fig_times = go.Figure()
    
    # Crear arrays de tiempo de simulación para el eje x
    sim_time = t[1:]  # Excluir el primer tiempo (k=0)
    
    # Agregar trazas de tiempo para cada filtro
    fig_times.add_trace(go.Scatter(
        x=sim_time, y=ekf_training_times,
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=2),
    ))
    
    fig_times.add_trace(go.Scatter(
        x=sim_time, y=ukf_training_times,
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=2),
    ))
    
    fig_times.add_trace(go.Scatter(
        x=sim_time, y=pf_training_times,
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=2),
    ))
    
    fig_times.update_layout(
        title={
            'text': 'Tiempos de Entrenamiento por Iteración',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo de Simulación (s)',
        yaxis_title='Tiempo de Entrenamiento (s)',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            type='log',  # Escala logarítmica para mejor visualización
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_times.show()
    
    # --- Gráfica de barras comparando tiempos medios ---
    fig_times_bar = go.Figure()
    
    filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
    mean_times_ms = [ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000]
    
    fig_times_bar.add_trace(go.Bar(
        x=filters,
        y=mean_times_ms,
        marker_color=['#1f77b4', '#2ca02c', '#d62728'],
        text=[f'{t:.3f} ms' for t in mean_times_ms],
        textposition='outside'
    ))
    
    fig_times_bar.update_layout(
        title={
            'text': 'Tiempo Promedio de Entrenamiento por Iteración',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Tiempo Promedio (ms)',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_times_bar.show()
    
    # --- Trayectoria 2D (Vista superior) ---
    fig_traj = go.Figure()
    
    # Trayectoria real
    fig_traj.add_trace(go.Scatter(
        x=x_true[:, 0], y=x_true[:, 1],
        mode='lines',
        name='Trayectoria Real',
        line=dict(color='#000000', width=thesis_config['line_width_true']),
    ))
    
    # Medidas ruidosas (puntos dispersos)
    fig_traj.add_trace(go.Scatter(
        x=y_measured[:, 0], y=y_measured[:, 1],
        mode='markers',
        name='Medidas Ruidosas',
        marker=dict(size=2, color='rgba(138, 43, 226, 0.9)', symbol='circle'),
    ))
    
    # Trayectorias estimadas
    fig_traj.add_trace(go.Scatter(
        x=x_est_ekf[:, 0], y=x_est_ekf[:, 1],
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
    ))
    
    fig_traj.add_trace(go.Scatter(
        x=x_est_ukf[:, 0], y=x_est_ukf[:, 1],
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
    ))
    
    fig_traj.add_trace(go.Scatter(
        x=x_est_pf[:, 0], y=x_est_pf[:, 1],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
    ))
    
    # Puntos de inicio y final
    fig_traj.add_trace(go.Scatter(
        x=[x_true[0, 0]], y=[x_true[0, 1]],
        mode='markers',
        name='Inicio',
        marker=dict(size=15, color='green', symbol='circle'),
    ))
    
    fig_traj.add_trace(go.Scatter(
        x=[x_true[-1, 0]], y=[x_true[-1, 1]],
        mode='markers',
        name='Final',
        marker=dict(size=15, color='red', symbol='square'),
    ))
    
    fig_traj.update_layout(
        title={
            'text': 'Trayectoria del Robot Diferencial - Configuración Paralela',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Posición x (m)',
        yaxis_title='Posición y (m)',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5,
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_width'],  # Cuadrado
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_traj.show()
    
    # --- Gráficas por Estado (con medidas ruidosas) ---
    states_info = [
        {'idx': 0, 'var': 'x', 'desc': 'Posición Horizontal', 'y_label': 'Posición x (m)'},
        {'idx': 1, 'var': 'y', 'desc': 'Posición Vertical', 'y_label': 'Posición y (m)'},
        {'idx': 2, 'var': 'θ', 'desc': 'Orientación', 'y_label': 'Ángulo θ (rad)'},
        {'idx': 3, 'var': 'v', 'desc': 'Velocidad Lineal', 'y_label': 'Velocidad v (m/s)'},
        {'idx': 4, 'var': 'ω', 'desc': 'Velocidad Angular', 'y_label': 'Velocidad ω (rad/s)'}
    ]
    
    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        # True state
        fig.add_trace(go.Scatter(
            x=t, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        # Noisy measurements
        fig.add_trace(go.Scatter(
            x=t, y=y_measured[:, i],
            mode='markers',
            name='Medidas',
            marker=dict(size=3, color='rgba(138, 43, 226, 0.9)'),  # Violeta (BlueViolet) con transparencia
            showlegend=True
        ))
        
        # EKF estimate
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ekf[:, i],
            mode='lines',
            name='EKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        # UKF estimate
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        # PF estimate
        fig.add_trace(go.Scatter(
            x=t, y=x_est_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Configuración Paralela',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()
    
    # --- Gráfica de errores acumulados ---
    fig_errors = go.Figure()
    
    # Calculate cumulative errors over time
    cumulative_error_ekf = np.zeros(n_steps)
    cumulative_error_ukf = np.zeros(n_steps)
    cumulative_error_pf = np.zeros(n_steps)
    
    for k in range(1, n_steps):
        error_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
        error_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
        error_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
        
        cumulative_error_ekf[k] = cumulative_error_ekf[k-1] + error_ekf
        cumulative_error_ukf[k] = cumulative_error_ukf[k-1] + error_ukf
        cumulative_error_pf[k] = cumulative_error_pf[k-1] + error_pf
    
    fig_errors.add_trace(go.Scatter(
        x=t, y=cumulative_error_ekf,
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=2),
    ))
    
    fig_errors.add_trace(go.Scatter(
        x=t, y=cumulative_error_ukf,
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=2),
    ))
    
    fig_errors.add_trace(go.Scatter(
        x=t, y=cumulative_error_pf,
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=2),
    ))
    
    fig_errors.update_layout(
        title={
            'text': 'Error Acumulado en el Tiempo - Norma Euclidiana',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title='Error Acumulado',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_errors.show()
    
    # --- Gráfica de compensación tiempo-precision (Pareto) ---
    fig_pareto = go.Figure()
    
    fig_pareto.add_trace(go.Scatter(
        x=[ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000],
        y=[mse_total_ekf, mse_total_ukf, mse_total_pf],
        mode='markers+text',
        text=['EKF', 'UKF', 'PF'],
        textposition='top center',
        marker=dict(
            size=15,
            color=['#1f77b4', '#2ca02c', '#d62728'],
            line=dict(width=2, color='DarkSlateGrey')
        ),
        name='Filtros'
    ))
    
    fig_pareto.update_layout(
        title={
            'text': 'Compensación Tiempo-Precisión (Frente de Pareto)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo Promedio por Iteración (ms)',
        yaxis_title='Error Cuadrático Medio Total (MSE)',
        xaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_pareto.show()
    
    print("\n✅ Visualización completa con configuración paralela y análisis de tiempos.")

    # ============================================================
    # 8) Additional Analysis
    # ============================================================
    
    print("\n" + "="*70)
    print("📈 ANÁLISIS ADICIONAL")
    print("="*70)
    
    # Analyze convergence
    convergence_window = 100  # Look at last 100 steps
    if n_steps > convergence_window:
        final_errors = {
            'EKF': np.mean((x_true[-convergence_window:, :] - x_est_ekf[-convergence_window:, :])**2, axis=0),
            'UKF': np.mean((x_true[-convergence_window:, :] - x_est_ukf[-convergence_window:, :])**2, axis=0),
            'PF': np.mean((x_true[-convergence_window:, :] - x_est_pf[-convergence_window:, :])**2, axis=0)
        }
        
        print(f"\nError promedio en los últimos {convergence_window} pasos:")
        for filter_name, errors in final_errors.items():
            print(f"\n{filter_name}:")
            print(f"  x: {errors[0]:.6f}, y: {errors[1]:.6f}, θ: {errors[2]:.6f}")
            print(f"  v: {errors[3]:.6f}, ω: {errors[4]:.6f}")
    
    # Resumen final
    print("\n" + "="*70)
    print("🎯 RESUMEN EJECUTIVO")
    print("="*70)
    print(f"Filtro más preciso: {best_filter}")
    print(f"Filtro más rápido: {'EKF-RHONN' if ekf_mean_time == fastest_mean else 'UKF-RHONN' if ukf_mean_time == fastest_mean else 'PF-RHONN'}")
    print(f"\nRecomendaciones:")
    print("1. Para aplicaciones en tiempo real: EKF-RHONN (velocidad)")
    print("2. Para máxima precisión: {best_filter}")
    print("3. Para robustez frente a no-linealidades fuertes: PF-RHONN")


Generando visualizaciones...



✅ Visualización completa con configuración paralela y análisis de tiempos.

📈 ANÁLISIS ADICIONAL

Error promedio en los últimos 100 pasos:

EKF:
  x: 0.000028, y: 0.000085, θ: 0.000791
  v: 0.000087, ω: 0.000033

UKF:
  x: 0.000031, y: 0.000071, θ: 0.000804
  v: 0.000124, ω: 0.000036

PF:
  x: 0.000053, y: 0.000068, θ: 0.000118
  v: 0.000192, ω: 0.000018

🎯 RESUMEN EJECUTIVO
Filtro más preciso: PF-RHONN
Filtro más rápido: EKF-RHONN

Recomendaciones:
1. Para aplicaciones en tiempo real: EKF-RHONN (velocidad)
2. Para máxima precisión: {best_filter}
3. Para robustez frente a no-linealidades fuertes: PF-RHONN


In [32]:
# Imprimir pesos finales de cada filtro
print("\n" + "="*70)
print("🔍 PESOS FINALES DE LAS REDES NEURONALES")
print("="*70)

print("\n--- EKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ekf.weights[i]}")

print("\n--- UKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ukf.weights[i]}")

print("\n--- PF-RHONN ---")
w_pf_final = pf.get_estimates()
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {w_pf_final[i]}")

print("\n" + "="*70)


🔍 PESOS FINALES DE LAS REDES NEURONALES

--- EKF-RHONN ---

Neurona 0 (x (pos)) - 4 pesos:
  [4.56131755 0.50465544 1.30275443 0.42081003]

Neurona 1 (y (pos)) - 3 pesos:
  [1.72719483 0.09838939 6.14689569]

Neurona 2 (θ (orient)) - 4 pesos:
  [ 1.53520117 -0.60612177  0.45602994  1.05163609]

Neurona 3 (v (lin vel)) - 5 pesos:
  [-0.07663913  1.34684187  0.52166548 -0.25876375  0.40662378]

Neurona 4 (ω (ang vel)) - 4 pesos:
  [ 2.19906877  4.85112764  1.55001816 -3.77884441]

--- UKF-RHONN ---

Neurona 0 (x (pos)) - 4 pesos:
  [  9.86190559 -13.68066625  22.70985227 -10.65393669]

Neurona 1 (y (pos)) - 3 pesos:
  [-1.39195341  0.82217971  8.07310805]

Neurona 2 (θ (orient)) - 4 pesos:
  [ 0.18343061 -1.01299396  2.8976653   1.16693872]

Neurona 3 (v (lin vel)) - 5 pesos:
  [-10.77212127  30.10075303 -19.4373452    0.54160942   0.44409038]

Neurona 4 (ω (ang vel)) - 4 pesos:
  [-0.16532803  8.96386261  1.25455269 -2.44916471]

--- PF-RHONN ---

Neurona 0 (x (pos)) - 4 pesos:
  [2.82